In [ ]:
#1. Импорт библиотек и выбор устройства (CPU/GPU)

import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Автоматический выбор GPU/CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используется устройство: {device}")


In [ ]:
#2. Настройка путей и проверка классов

import kagglehub

# 1. Скачиваем/проверяем датасет. kagglehub сам вернет ПРАВИЛЬНЫЙ путь 
# для текущего компьютера (например, C:\Users\... на Windows или /root/.cache/... в Colab)
dataset_path = kagglehub.dataset_download("puneet6060/intel-image-classification")
print("Абсолютный путь к датасету на этой машине:", dataset_path)

# 2. Собираем пути к папкам обучения и теста с помощью os.path.join.
# Это защищает от ошибок слэшей (Windows использует backslash \, а Linux/Colab использует slash /)
TRAIN_DIR = os.path.join(dataset_path, 'seg_train', 'seg_train')
TEST_DIR = os.path.join(dataset_path, 'seg_test', 'seg_test')

# 3. Проверка работоспособности путей
if os.path.exists(TRAIN_DIR) and os.path.exists(TEST_DIR):
    classes = sorted(os.listdir(TRAIN_DIR))
    print(f"\n[УСПЕХ] Пути настроены корректно!")
    print(f"Обнаружено классов: {len(classes)} -> {classes}")
else:
    print("\n[ОШИБКА] Что-то пошло не так со структурой папок внутри скачанного архива.")
    print("Содержимое корневой папки датасета:", os.listdir(dataset_path))




In [ ]:
#3. Визуальный анализ данных (EDA) — Вывод случайных изображений

fig, axes = plt.subplots(len(classes), 3, figsize=(10, 15))
for idx, class_name in enumerate(classes):
    class_path = os.path.join(TRAIN_DIR, class_name)
    images = os.listdir(class_path)
    random_imgs = random.sample(images, 3)
    
    for i, img_name in enumerate(random_imgs):
        img_path = os.path.join(class_path, img_name)
        img = Image.open(img_path)
        axes[idx, i].imshow(img)
        axes[idx, i].axis('off')
        if i == 1:
            axes[idx, i].set_title(class_name, fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
#4. Определение трансформаций (Продвинутая аугментация)

IMG_SIZE = 150  # Родной размер датасета Intel
BATCH_SIZE = 64

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),                        # Отражение по горизонтали
    transforms.RandomRotation(degrees=15),                         # Случайный поворот
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2), # Изменение цвета/контраста
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # ImageNet нормализация
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1), value='random') # Случайные маски-вырезы
])

test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [ ]:
#5. Создание Dataset и DataLoaders

# Автоматически определяет количество доступных ядер CPU

import multiprocessing
num_workers = min(4, multiprocessing.cpu_count()) 

# Проверяем, доступна ли видеокарта прямо сейчас
is_cuda = torch.cuda.is_available()

# Инициализация DataLoaders с динамической настройкой pin_memory
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=num_workers, 
    pin_memory=is_cuda  # Будет True только если есть видеокарта, иначе False
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=num_workers, 
    pin_memory=is_cuda
)

print(f"Размер тренировочной выборки: {len(train_dataset)}")
print(f"Размер тестовой выборки: {len(test_dataset)}")
print(f"Параметр pin_memory автоматически установлен в: {is_cuda}")



In [ ]:
#6. Создание архитектуры Multi-Resolution Residual CNN
class MultiScaleResidualBlock(nn.Module):

    #Блок, обрабатывающий признаки параллельно на разных разрешениях (фильтры 3x3 и 5x5)
    #с добавлением остаточного (Residual) соединения.
    
    def __init__(self, in_channels, out_channels):
        super(MultiScaleResidualBlock, self).__init__()
        mid_channels = out_channels // 2
        
        # Ветка 1: Фильтр 3x3 для мелких деталей и текстур
        self.branch3x3 = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU()
        )
        
        # Ветка 2: Фильтр 5x5 для крупных форм и контуров (линия горизонта, горы)
        self.branch5x5 = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=5, padding=2),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU()
        )
        
        # Перемешивание каналов из разных веток
        self.conv1x1 = nn.Conv2d(out_channels, out_channels, kernel_size=1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        
        # Shortcut (соединение-пропуск) для выравнивания размерностей
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out3x3 = self.branch3x3(x)
        out5x5 = self.branch5x5(x)
        out = torch.cat([out3x3, out5x5], dim=1) # Конкатенация веток
        out = self.bn(self.conv1x1(out))
        out += self.shortcut(x)                  # Сложение с оригинальным входом
        return self.relu(out)


class AdvancedMultiResCNN(nn.Module):
    def __init__(self, num_classes=6):
        super(AdvancedMultiResCNN, self).__init__()
        
        # Входной тензор: [B, 3, 150, 150]
        self.init_conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        
        # Каскад слоев со сжатием пространственного разрешения
        self.layer1 = MultiScaleResidualBlock(32, 64)    # Размеры: [B, 64, 150, 150]
        self.pool1 = nn.MaxPool2d(2, 2)                  # Размеры: [B, 64, 75, 75]
        
        self.layer2 = MultiScaleResidualBlock(64, 128)   # Размеры: [B, 128, 75, 75]
        self.pool2 = nn.MaxPool2d(2, 2)                  # Размеры: [B, 128, 37, 37]
        
        self.layer3 = MultiScaleResidualBlock(128, 256)  # Размеры: [B, 256, 37, 37]
        self.pool3 = nn.MaxPool2d(2, 2)                  # Размеры: [B, 256, 18, 18]
        
        self.layer4 = MultiScaleResidualBlock(256, 512)  # Размеры: [B, 512, 18, 18]
        self.pool4 = nn.MaxPool2d(2, 2)                  # Размеры: [B, 512, 9, 9]
        
        # Адаптивный глобальный пулинг сжимает 9x9 карту в вектор 1x1
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))  # Размеры: [B, 512, 1, 1]
        
        # Классификатор (Полносвязный блок)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        x = self.init_conv(x)
        x = self.pool1(self.layer1(x))
        x = self.pool2(self.layer2(x))
        x = self.pool3(self.layer3(x))
        x = self.pool4(self.layer4(x))
        x = self.global_pool(x)
        return self.classifier(x)

# Инициализация модели
model = AdvancedMultiResCNN(num_classes=len(classes)).to(device)


In [ ]:
#7. Конфигурация параметров функции потерь и Оптимизатора

criterion = nn.CrossEntropyLoss()

# Оптимизатор AdamW лучше справляется с L2-регуляризацией (weight decay)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)

EPOCHS = 15
# Косинусный планировщик плавно снижает LR до нуля к концу обучения
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# Словaрь для записи метрик
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}


In [ ]:
#8. Цикл обучения модели (Training Loop)

for epoch in range(EPOCHS):
    # --- Режим Обучения ---
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
    epoch_train_loss = running_loss / len(train_loader.dataset)
    epoch_train_acc = correct_train / total_train
    
    # --- Режим Тестирования (Валидации) ---
    model.eval()
    running_val_loss, correct_val, total_val = 0.0, 0, 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    epoch_val_loss = running_val_loss / len(test_loader.dataset)
    epoch_val_acc = correct_val / total_val
    
    # Шаг шедулера в концу эпохи
    scheduler.step()
    
    # Сохраняем логи
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    
    print(f"Эпоха [{epoch+1}/{EPOCHS}] | "
          f"Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc*100:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc*100:.2f}%")


In [ ]:
#9. Оценка результатов, графики и Матрица ошибок

# 9.1 Построение графиков обучения

epochs_range = range(1, EPOCHS + 1)
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history['train_loss'], label='Train Loss', color='royalblue', lw=2)
plt.plot(epochs_range, history['val_loss'], label='Val Loss', color='orange', lw=2)
plt.title('Зависимость функции потерь (Loss) от эпохи')
plt.xlabel('Эпоха')
plt.ylabel('Loss')
plt.grid(True, linestyle='--')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history['train_acc'], label='Train Acc', color='royalblue', lw=2)
plt.plot(epochs_range, history['val_acc'], label='Val Acc', color='orange', lw=2)
plt.title('Зависимость точности (Accuracy) от эпохи')
plt.xlabel('Эпоха')
plt.ylabel('Accuracy')
plt.grid(True, linestyle='--')
plt.legend()
plt.show()

# 9.2 Сбор финальных предсказаний для метрик
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# Итоговый текстовый отчет (Precision, Recall, F1)
print("\n=== Итоговый отчет о классификации ===")
print(classification_report(all_labels, all_preds, target_names=classes))

# Отрисовка матрицы ошибок (Confusion Matrix)
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Матрица ошибок (Confusion Matrix)')
plt.ylabel('Истинные классы')
plt.xlabel('Предсказанные классы')
plt.show()
